# Notebook 1 – Setup & Data Preparation

**Purpose:** Install all required packages, verify the AWS/SageMaker environment, configure your S3 bucket, upload the ASL Alphabet dataset to S3, run the stratified train/val/test split, and write `config.json` so the other notebooks know where everything lives.

**Run this notebook FIRST before running any other notebook.**

---
### What this notebook does
1. Installs Python dependencies (TensorFlow, MediaPipe, OpenCV, scikit-learn …)
2. Confirms AWS credentials and SageMaker role are working
3. Creates (or reuses) an S3 bucket and sets up folder prefixes
4. Walks you through uploading the Kaggle ASL Alphabet dataset to S3
5. Runs a stratified 70 / 15 / 15 split and re-uploads the organised splits
6. Writes `config.json` consumed by Notebooks 2 and 3

## Step 1 – Install Required Packages

> This may take 3–5 minutes on first run.

In [ ]:
import subprocess, sys

# Install numpy first so subsequent packages cannot pull in numpy 2.x
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24,<2.0"])
    print("  ✅ numpy (pre-pinned to <2.0)")
except subprocess.CalledProcessError as e:
    print(f"  ⚠️  numpy pre-pin failed: {e}")

packages = [
    "tensorflow>=2.13,<3.0",
    "mediapipe==0.10.33",          # pinned: 0.10.30-0.10.33 are the versions
                                   # available on SageMaker; 0.10.9 does not exist
    "numpy>=1.24,<2.0",            # avoid NumPy 2.x ABI breakage with MediaPipe
    "opencv-python-headless>=4.8,<4.10",  # 4.10+ requires numpy>=2, incompatible with mediapipe
    "scikit-learn>=1.3",
    "matplotlib>=3.7",
    "seaborn>=0.12",
    "pandas>=2.0",
    "Pillow>=10.0",
    "tabulate>=0.9",
    "boto3>=1.26",
    "sagemaker>=2.180",
]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✅ {pkg.split('=')[0].split('>')[0].split('<')[0]}")
    except subprocess.CalledProcessError as e:
        print(f"  ⚠️  {pkg} failed: {e}  (continuing …)")

print("\n✅ Package installation complete.")

## Step 1b – Verify MediaPipe Installation

Run this cell to confirm which MediaPipe API mode is active.
The helper module `mediapipe_utils.py` (included in this repo) supports:
- **Tasks API** (mediapipe 0.10.30+) — requires downloading a small model file
- **Solutions API** (older builds that still expose `mp.solutions`)
- **Fallback** — CNN approaches still work; Landmark MLP is skipped gracefully

In [ ]:
# Verify MediaPipe and display compatibility info
import importlib, sys

try:
    import mediapipe as mp
    print(f"MediaPipe version : {mp.__version__}")
except ImportError:
    print("❌ MediaPipe not importable — check installation above")

# Show which API mode will be used by the training notebook
try:
    from mediapipe_utils import print_mediapipe_info
    print_mediapipe_info()
except ImportError:
    print("⚠️  mediapipe_utils.py not found in current directory.")
    print("   Make sure mediapipe_utils.py is in the same folder as the notebooks.")


## Step 2 – Verify AWS Environment

In [ ]:
import boto3, sagemaker, os

try:
    session     = sagemaker.Session()
    role        = sagemaker.get_execution_role()
    region      = boto3.session.Session().region_name
    account_id  = boto3.client("sts").get_caller_identity()["Account"]
    print(f"✅ SageMaker role  : {role}")
    print(f"   Region          : {region}")
    print(f"   Account ID      : {account_id}")
except Exception as e:
    print(f"⚠️  Could not auto-detect SageMaker role: {e}")
    print("   Running outside SageMaker Studio — AWS calls will use your local credentials.")
    region = boto3.session.Session().region_name or "us-east-1"
    print(f"   Region set to   : {region}")

## Step 3 – Configure S3 Bucket

Edit the variables below, then run the cell.  
- `BUCKET_NAME` – your S3 bucket (will be created if it doesn't exist)  
- `PREFIX` – top-level folder inside the bucket

In [ ]:
import boto3, json

# ── EDIT THESE ─────────────────────────────────────────────────────────────────
BUCKET_NAME  = "asl-recognition-bucket"   # change to a globally unique name
PREFIX       = "asl-alphabet"
# ──────────────────────────────────────────────────────────────────────────────

s3  = boto3.client("s3")
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Bucket already exists: s3://{BUCKET_NAME}")
except s3.exceptions.ClientError as e:
    code_str = e.response["Error"]["Code"]
    if code_str in ("404", "NoSuchBucket"):
        try:
            if boto3.session.Session().region_name == "us-east-1":
                s3.create_bucket(Bucket=BUCKET_NAME)
            else:
                s3.create_bucket(
                    Bucket=BUCKET_NAME,
                    CreateBucketConfiguration={"LocationConstraint": boto3.session.Session().region_name}
                )
            print(f"✅ Created bucket: s3://{BUCKET_NAME}")
        except Exception as create_err:
            print(f"❌ Could not create bucket: {create_err}")
            raise
    else:
        raise

print(f"   S3 prefix: {PREFIX}/")

## Step 4 – Upload the ASL Alphabet Dataset

The project uses the **Kaggle ASL Alphabet Dataset** (87,000 images, 29 classes A-Z + del/nothing/space).

### Download instructions (choose one option)

**Option A – Kaggle CLI (recommended inside SageMaker)**
```bash
pip install kaggle
# Upload your kaggle.json API token first, then:
kaggle datasets download -d grassknoted/asl-alphabet --unzip -p /tmp/asl_raw
```

**Option B – Manual upload**
1. Download from https://www.kaggle.com/datasets/grassknoted/asl-alphabet
2. Upload the zip to this JupyterLab session via the file browser
3. Set `LOCAL_ZIP_PATH` below to the path of the zip file

After download/unzip, the expected structure is:
```
/tmp/asl_raw/asl_alphabet_train/   (or similar top-level folder)
    A/   B/   C/ … Z/   del/   nothing/   space/
```

In [ ]:
# ── Configure dataset source ───────────────────────────────────────────────────
import os, zipfile, shutil

# Set to your actual local path if doing a manual upload:
LOCAL_ZIP_PATH  = None          # e.g.  "/home/ec2-user/asl-alphabet.zip"
KAGGLE_DOWNLOAD = True          # set False if using LOCAL_ZIP_PATH

RAW_DIR = "/tmp/asl_raw"

if KAGGLE_DOWNLOAD:
    try:
        import subprocess
        subprocess.check_call(["pip", "install", "-q", "kaggle"])
        os.makedirs(RAW_DIR, exist_ok=True)
        subprocess.check_call([
            "kaggle", "datasets", "download",
            "-d", "grassknoted/asl-alphabet",
            "--unzip", "-p", RAW_DIR
        ])
        print(f"✅ Dataset downloaded to {RAW_DIR}")
    except Exception as e:
        print(f"❌ Kaggle download failed: {e}")
        print("   → Ensure kaggle.json is in ~/.kaggle/ or set KAGGLE_DOWNLOAD=False and use LOCAL_ZIP_PATH.")
        raise

elif LOCAL_ZIP_PATH:
    os.makedirs(RAW_DIR, exist_ok=True)
    print(f"Extracting {LOCAL_ZIP_PATH} …")
    try:
        with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zf:
            zf.extractall(RAW_DIR)
        print(f"✅ Extracted to {RAW_DIR}")
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        raise
else:
    print("⚠️  Neither Kaggle download nor a zip path was provided.")
    print("   Set KAGGLE_DOWNLOAD=True  OR  set LOCAL_ZIP_PATH to your zip file.")

In [ ]:
# ── Locate the class sub-folders ───────────────────────────────────────────────
import os

def find_class_root(base_dir):
    """Walk the tree to find the directory that directly contains class sub-folders."""
    expected_classes = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ") | {"del", "nothing", "space"}
    for root, dirs, files in os.walk(base_dir):
        found = {d for d in dirs if d in expected_classes}
        if len(found) >= 26:          # found most expected classes
            return root
    return None

CLASS_ROOT = find_class_root(RAW_DIR)
if CLASS_ROOT:
    classes = sorted(os.listdir(CLASS_ROOT))
    print(f"✅ Found class root: {CLASS_ROOT}")
    print(f"   Classes ({len(classes)}): {classes}")
else:
    print("❌ Could not locate class sub-folders.")
    print(f"   Contents of {RAW_DIR}:")
    for item in os.listdir(RAW_DIR):
        print(f"     {item}")
    raise RuntimeError(f"Expected class sub-folders not found in {RAW_DIR}")

## Step 5 – Stratified Train / Val / Test Split (70 / 15 / 15)

In [ ]:
import os, shutil, random
from sklearn.model_selection import train_test_split

SPLIT_DIR   = "/tmp/asl_split"
RANDOM_SEED = 42

CLASS_NAMES = sorted([d for d in os.listdir(CLASS_ROOT) if os.path.isdir(os.path.join(CLASS_ROOT, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# Collect all paths + labels
all_paths, all_labels = [], []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(CLASS_ROOT, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(cls)

print(f"Total images: {len(all_paths)}")

# Stratified split
X_tv, X_test, y_tv, y_test = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=RANDOM_SEED, stratify=all_labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/(0.70+0.15), random_state=RANDOM_SEED, stratify=y_tv)

print(f"Split → train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

# Copy into class sub-folders
for split_name, paths, labels in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    for src, lbl in zip(paths, labels):
        dest_dir = os.path.join(SPLIT_DIR, split_name, lbl)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(src, os.path.join(dest_dir, os.path.basename(src)))

print("✅ Split complete.")

## Step 6 – Upload Split Dataset to S3

In [ ]:
import boto3, os
from pathlib import Path

s3 = boto3.client("s3")

def upload_dir(local_dir, bucket, s3_prefix):
    files = list(Path(local_dir).rglob("*"))
    files = [f for f in files if f.is_file()]
    print(f"Uploading {len(files)} files from {local_dir} …")
    for i, fpath in enumerate(files):
        rel  = fpath.relative_to(local_dir)
        key  = f"{s3_prefix}/{rel}".replace(os.sep, "/")
        try:
            s3.upload_file(str(fpath), bucket, key)
        except Exception as e:
            print(f"  ❌ Failed to upload {fpath}: {e}")
        if (i + 1) % 5000 == 0:
            print(f"  … {i+1}/{len(files)} uploaded")
    print(f"  ✅ Done: s3://{bucket}/{s3_prefix}/")

for split in ["train", "val", "test"]:
    upload_dir(
        local_dir   = os.path.join(SPLIT_DIR, split),
        bucket      = BUCKET_NAME,
        s3_prefix   = f"{PREFIX}/{split}",
    )

## Step 7 – Verify S3 Upload & Write config.json

In [ ]:
import boto3, json

s3 = boto3.client("s3")

split_counts = {}
for split in ["train", "val", "test"]:
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=BUCKET_NAME, Prefix=f"{PREFIX}/{split}/")
    count = sum(page.get("KeyCount", 0) for page in pages)
    split_counts[split] = count
    print(f"  s3://{BUCKET_NAME}/{PREFIX}/{split}/  →  {count} objects")

config = {
    "bucket":       BUCKET_NAME,
    "prefix":       PREFIX,
    "region":       boto3.session.Session().region_name,
    "class_names":  CLASS_NAMES,
    "num_classes":  NUM_CLASSES,
    "split_counts": split_counts,
    "s3_train_uri": f"s3://{BUCKET_NAME}/{PREFIX}/train",
    "s3_val_uri":   f"s3://{BUCKET_NAME}/{PREFIX}/val",
    "s3_test_uri":  f"s3://{BUCKET_NAME}/{PREFIX}/test",
    "models_prefix": f"{PREFIX}/models",
    "results_prefix": f"{PREFIX}/results",
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("\n✅ config.json written:")
print(json.dumps(config, indent=2))

---
## ✅ Notebook 1 Complete

Your S3 bucket now contains:

```
s3://<BUCKET>/<PREFIX>/
    train/   A/ … Z/ del/ nothing/ space/   (~60 900 images)
    val/     A/ … Z/ del/ nothing/ space/   (~13 050 images)
    test/    A/ … Z/ del/ nothing/ space/   (~13 050 images)
```

A `config.json` has been written to this directory and will be consumed by **Notebook 2** (training) and **Notebook 3** (evaluation).

**Next step → open `02_train_models.ipynb`**